# Pendahuluan

Di dalam material ini, kamu akan membuat dashboard prediksi menggunakan Streamlit untuk dataset klasifikasi Breast Cancer (diagnosis ganas atau jinak). Dashboard ini memungkinkan pengguna memasukkan nilai fitur (manual lewat slider atau unggah CSV) dan mendapatkan prediksi dari model yang sudah dilatih pada Capstone II.

**Langkah 1: Instalasi dan Persiapan**

Pastikan modul streamlit, pandas, dan pillow sudah terpasang (`pip install streamlit pandas pillow scikit-learn`). Pastikan file berikut ada di folder yang sama dengan skrip:
- `generate_breast_cancer.pkl` (hasil Capstone II)
- `data.csv` (hasil Capstone I, dipakai untuk menentukan rentang slider)
- gambar header, misalnya `breast-cancer.jpg` (opsional; kalau tidak ada, gambar dilewati)

**Langkah 2: Import Modul**

Import semua modul yang dibutuhkan:

In [1]:
import os
import streamlit as st
import pandas as pd
import pickle
import time
from PIL import Image

**Langkah 3: Konfigurasi Halaman dan Judul**

Konfigurasi halaman Streamlit dan tulis judul dashboard:

In [2]:
st.set_page_config(page_title="Halaman Modelling", layout="wide")
st.write("""
# Welcome to my machine learning dashboard

This dashboard created by : Azzahra Putri Kamilah""")

2026-11-07 11:04:22.955 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:22.960 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:35.426 
  command:

    streamlit run C:\Users\USER\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-11-07 11:04:35.430 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:35.446 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


**Langkah 4: Pilihan Kategori**

Tambahkan pilihan kategori untuk membuka dashboard prediksi:

In [3]:
add_selectitem = st.sidebar.selectbox("Want to open about?", ("Breast Cancer!",))

2026-11-07 11:04:35.722 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:35.742 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:35.744 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:35.754 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:35.773 Session state does not function when running a script without `streamlit run`
2026-11-07 11:04:35.794 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:35.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:35.836 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Ketentuan.

Jika pengguna memilih "Breast Cancer!", dashboard akan memprediksi diagnosis kanker payudara (ganas atau jinak) berdasarkan input yang diberikan. Pengguna dapat mengunggah file CSV dengan data input atau mengisi manual. Setiap fitur diatur melalui slider dengan rentang nilai dari data pelatihan. Setelah input selesai, tekan tombol Predict! untuk melihat hasil prediksi.

**Langkah 5: Fungsi untuk Prediksi Breast Cancer**

Buat fungsi breast_cancer() yang menampilkan aplikasi prediksi. Data yang digunakan berasal dari dataset Breast Cancer Wisconsin (Diagnostic). Pengguna dapat memberikan input melalui slider atau mengunggah file CSV. Setelah input diberikan, model yang sudah disimpan dimuat dengan pickle, prediksi dijalankan, dan hasilnya ditampilkan.

In [4]:
MODEL_PATH = "generate_breast_cancer.pkl"
DATA_PATH = "data.csv"
IMAGE_PATH = "breast-cancer.jpg"

@st.cache_resource
def load_model(path):
    with open(path, 'rb') as file:
        return pickle.load(file)

@st.cache_data
def load_reference(path):
    return pd.read_csv(path, index_col=0)

def breast_cancer():
    st.write("""
    This app predicts the **Breast Cancer** diagnosis (Malignant or Benign)

    Data obtained from the [Breast Cancer Wisconsin (Diagnostic) dataset](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) by UCIML.
    """)
    loaded_model = load_model(MODEL_PATH)
    fitur = list(loaded_model.feature_names_in_)   # fitur & urutannya sama dengan saat training
    ref = load_reference(DATA_PATH)

    st.sidebar.header('User Input Features:')
    # Collects user input features into dataframe
    uploaded_file = st.sidebar.file_uploader("Upload your input CSV file", type=["csv"])
    if uploaded_file is not None:
        input_df = pd.read_csv(uploaded_file)
        kurang = [f for f in fitur if f not in input_df.columns]
        if kurang:
            st.error(f"Kolom berikut tidak ada di file CSV: {kurang}")
            st.stop()
        input_df = input_df[fitur]
    else:
        def user_input_features():
            st.sidebar.header('Manual Input')
            data = {}
            for f in fitur:
                lo, hi, med = float(ref[f].min()), float(ref[f].max()), float(ref[f].median())
                data[f] = st.sidebar.slider(f, lo, hi, med, step=(hi - lo) / 200)
            features = pd.DataFrame(data, index=[0])
            return features

        input_df = user_input_features()

    if os.path.exists(IMAGE_PATH):
        img = Image.open(IMAGE_PATH)
        st.image(img, width=500)

    if st.sidebar.button('Predict!'):
        df = input_df
        st.write(df)
        prediction = loaded_model.predict(df)
        proba = loaded_model.predict_proba(df)[:, 1]
        st.subheader('Prediction: ')
        hasil = ['Benign (Jinak)' if p == 0 else 'Malignant (Ganas)' for p in prediction]
        with st.spinner('Wait for it...'):
            time.sleep(2)
            for h, p in zip(hasil, proba):
                st.success(f"Prediction of this app is {h} (probabilitas ganas: {p:.2%})")

2026-11-07 11:04:36.314 No runtime found, using MemoryCacheStorageManager


Kita mendefinisikan fungsi breast_cancer() untuk membangun aplikasi prediksi kanker payudara. Model yang sudah di-training disimpan dalam file generate_breast_cancer.pkl berupa pipeline (scaling + model), sehingga input mentah langsung bisa diprediksi. Nama fitur dan urutannya dibaca langsung dari model, dan rentang slider dari data.csv. Hasil prediksi ditampilkan dengan st.success.

**Langkah 6: Menjalankan Fungsi Berdasarkan Pilihan**

Gunakan pernyataan if untuk menjalankan fungsi yang sesuai berdasarkan pilihan pengguna:

In [5]:
def main():
    if add_selectitem == "Breast Cancer!":
        breast_cancer()

**Langkah 7: Menjalankan Aplikasi**

Terakhir, panggil fungsi main(). Simpan seluruh kode di atas ke file `app.py`, lalu jalankan di terminal dengan `streamlit run app.py`.

In [6]:
if __name__ == "__main__":
    main()

2026-11-07 11:04:37.293 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:37.308 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:37.310 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:37.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:37.321 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:37.342 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:37.345 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-07 11:04:37.928 Thread 'Thread-3': missing ScriptRunContext! This warning can be ignored when running in bare 